In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets
sklearn.set_config(display='text')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

## 캘리포니아 집값 데이터를 불러와서 스케일링 한다.

캘리포니아 집값 데이터를 불러온다.

In [2]:
raw_data = datasets.fetch_california_housing()
print(raw_data)

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8)), 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)), 'frame': None, 'target_names': ['MedHouseVal'], 'feature_names': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'], 'DESCR': '.. _california_housing_dataset

캘리포니아 집값 데이터에서 피처와 피처에 따른 레이블을 얻어온다.

In [3]:
xData = raw_data.data # 피처
yData = raw_data.target # 피처에 따른 레이블
print(xData.shape, yData.shape)

(20640, 8) (20640,)


학습 데이터와 테스트 데이터로 분할한다.

In [4]:
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(15480, 8) (15480,) (5160, 8) (5160,)


데이터를 스케일링(표준화) 한다.  
학습 데이터는 스케일링 실행 후 적용하고 텍스트 데이터는 학습 데이터로 스케일링된 결과에 적용만 시킨다.

In [5]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

## 캘리포니아 집값을 예측하는 일반 선형 회귀 모델을 생성하고 학습시킨다.

일반 선형 회귀 알고리즘을 하용하기 위해 LinearRegression을 import 한다.

In [6]:
from sklearn.linear_model import LinearRegression

선형 회귀 모델에 학습 데이터 피처(x_train)와 학습 데이터 피처에 따른 레이블(y_train)을 넘겨서 학습시킨다.

In [7]:
# model_linear = LinearRegression()
# model_linear.fit(x_train, y_train)
model_linear = LinearRegression().fit(x_train, y_train)

가중치(회귀 계수)와 바이어스(상수항)를 출력한다.  
선형 회귀 모델에서 coef_ 속성으로 가중치(피처의 개수)와 intercept_ 속성으로 바이어스(무조건 1개)를 얻어온다.

In [8]:
print('가중치:\n', model_linear.coef_, sep='')
print('바이어스:\n', model_linear.intercept_, sep='')

가중치:
[ 0.83189945  0.1209374  -0.26175157  0.30405212 -0.00873559 -0.02984442
 -0.89236538 -0.86385031]
바이어스:
2.074372627906947


## 캘리포니아 집값을 예측하는 라쏘(lasso) 선형 회귀 모델을 생성하고 학습시킨다. 

라쏘 선형 회귀 분석(L1 제약식 사용)

라쏘를 사용하는 이유는 `불필요한 변수를 자동으로 제거`하여 모델을 `단순`하고 해석하기 쉽게 만들기 위해서 사용한다.  
L1 제약식은 머신러닝, 특히 선형 회귀 모델에서 과적합을 막고 일반화 성능을 높이기 위해 사용하는 가중치 규제 기법으로 모델이 학습할 때 가중치가 너무 커지지 않도록 제약하는 방식이며, 수학적으로는 가중치들의 `절대값의 합`을 제한 조건으로 한다.

L1 제약식의 특징은 중요도가 떨어지거나 불필요한 피처의 가중치를 `0`으로 만든다.

라쏘 선형 회귀 알고리즘을 하용하기 위해 Lasso를 import 한다.

In [9]:
from sklearn.linear_model import Lasso

라쏘 선형 회귀 모델의 `alpha`라는 인자를 이용해서 `규제 강도를 지정`해서 모델을 만든다.  
alpha는 규제 강도를 조절하는 하이퍼파라미터로 기본값은 1.0이고 값이 클 수록 강한 제약을 의미한다. 수학적 범위는 0이상 무한대까지이고 실무적 범위는 로그 스케일(0.001, 0.01, 0.1, 1, 10, 100, ...) 값을 사용한다.  
alpha 값을 0으로 지정하면 규제를 전혀 적용하지 않는다는 의미로 일반 선형 회귀와 똑같이 동작한다.

In [23]:
# model_lasso = Lasso(alpha=0)
# model_lasso.fit(x_train, y_train)
model_lasso = Lasso(alpha=0.01).fit(x_train, y_train)

In [24]:
print('가중치:\n', model_lasso.coef_, sep='')
print('바이어스:\n', model_lasso.intercept_, sep='')

가중치:
[ 0.77897391  0.12866563 -0.11947889  0.16185133 -0.         -0.02249526
 -0.78774759 -0.75273912]
바이어스:
2.0743726279069525


## 캘리포니아 집값을 예측하는 릿지(ridge) 선형 회귀 모델을 생성하고 학습시킨다.

릿지 선형 회귀 분석(L2 제약식 사용)

L2 제약식은 머신러닝, 특히 선형 회귀 모델에서 과적합을 방지하고 일반화 성능을 높이기 위해 사용하는 가중치 규제 기법으로 모델이 학습할 때 가중치가 너무 커지지 않도록 제약을 거는 방식이며, 수학적으로는 가중치들의 `제곱의 합`을 제한 조건으로 한다.

L2 제약식의 특징은 L1 제약식과 마찬가지로 중요도가 떨어지거나 불필요한 피처의 가중치를 0으로 만든다.

릿지 선형 회귀 알고리즘을 하용하기 위해 Ridge를 import 한다.

In [25]:
from sklearn.linear_model import Ridge

릿지 선형 회귀 모델의 alpha라는 인자를 이용해서 규제 강도를 지정해서 모델을 만든다. 사용 방법은 라쏘 선형 회귀 모델과 같다.

In [49]:
# model_ridge = Ridge(alpha=1)
# model_ridge.fit(x_train, y_train)
model_ridge = Ridge(alpha=1).fit(x_train, y_train)

In [50]:
print('가중치:\n', model_ridge.coef_, sep='')
print('바이어스:\n', model_ridge.intercept_, sep='')

가중치:
[ 0.83186352  0.12102542 -0.26157085  0.30381865 -0.00870202 -0.02985494
 -0.89154458 -0.86302194]
바이어스:
2.074372627906947


## 캘리포니아 집값을 예측하는 엘라스틱 넷(elastic net) 선형 회귀 모델을 생성하고 학습시킨다.

엘라스틱 넷 선형 회귀 분석(L1, L2 제약식 모두 사용)

엘라스틱 넷은 라쏘(L1 규제)와 릿지(L2 규제)의 장점을 결합한 하이브리드 선형 회귀 규제 모델로 라쏘와 릿지가 각각 가진 한계점을 보완하기 위해 만들어졌으며, 특히 변수 간 상관관계가 높고 데이터가 복잡할 때 사용한다.

엘라스틱 넷 선형 회귀 알고리즘을 하용하기 위해 ElasticNet을 import 한다.

In [51]:
from sklearn.linear_model import ElasticNet

엘라스틱 넷 선형 회귀 모델의 alpha라는 인자를 이용해서 규제 강도를 지정해서 모델을 만든다.  
alpha에는 L1 제약식의 크기와 L2 제약식의 크기의 합을 지정하고 `l1_ratio`에 alpha에서 지정한 값에서 L1 제약이 차지하는 비율을 지정한다.  
l1_ratio는 비율이므로 0부터 1사이의 값을 지정해야 하며, `0`을 지정하면 L1 제약식이 사용되지 않고 L2 제약만 사용하는 `릿지` 선형 회귀 분석을 의미하고, `1`을 지정하면 L2 제약식이 사용되지 않고 L1 제약만 사용하는 `라쏘` 선형 회귀 분석이 된다.

In [60]:
# model_elastic = ElasticNet(alpha=0.01, l1_ratio=0.01)
# model_elastic.fit(x_train, y_train)
model_elastic = ElasticNet(alpha=0.01, l1_ratio=0.01).fit(x_train, y_train)

In [61]:
print('가중치:\n', model_elastic.coef_, sep='')
print('바이어스:\n', model_elastic.intercept_, sep='')

가중치:
[ 0.8239746   0.13234248 -0.23241126  0.26811823 -0.00424703 -0.03112002
 -0.78337789 -0.75358665]
바이어스:
2.074372627906953


## 학습된 모델들로 테스트 데이터를 예측한다.

In [63]:
predict_linear = model_linear.predict(x_test)
predict_lasso = model_lasso.predict(x_test)
predict_ridge = model_ridge.predict(x_test)
predict_elastic = model_elastic.predict(x_test)

for i in range(len(predict_linear))[:5]:
    print('일반: {:8.5f}, 라쏘: {:8.5f}, 릿지: {:8.5f}, 엘라: {:8.5f}'.format(predict_linear[i], predict_lasso[i], predict_ridge[i], predict_elastic[i]))

일반:  2.27826, 라쏘:  2.27774, 릿지:  2.27811, 엘라:  2.25850
일반:  2.79607, 라쏘:  2.83514, 릿지:  2.79618, 엘라:  2.81142
일반:  1.90887, 라쏘:  1.96669, 릿지:  1.90916, 엘라:  1.94769
일반:  1.02576, 라쏘:  1.07487, 릿지:  1.02638, 엘라:  1.10835
일반:  2.95860, 라쏘:  2.81830, 릿지:  2.95813, 엘라:  2.89183


## 학습된 모델을 평가한다.

평균 절대값 오차

In [65]:
print('일반:', mean_absolute_error(y_test, predict_linear))
print('라쏘:', mean_absolute_error(y_test, predict_lasso))
print('릿지:', mean_absolute_error(y_test, predict_ridge))
print('엘라:', mean_absolute_error(y_test, predict_elastic))

일반: 0.5368950735045218
라쏘: 0.5409381699751551
릿지: 0.5368952963837784
엘라: 0.5380423238460021


평균 제곱 오차

In [66]:
print('일반:', mean_squared_error(y_test, predict_linear))
print('라쏘:', mean_squared_error(y_test, predict_lasso))
print('릿지:', mean_squared_error(y_test, predict_ridge))
print('엘라:', mean_squared_error(y_test, predict_elastic))

일반: 0.5404128061709079
라쏘: 0.547654376568303
릿지: 0.5404220132440147
엘라: 0.5435117084883777


평균 제곱근 오차

In [67]:
print('일반:', np.sqrt(mean_squared_error(y_test, predict_linear)))
print('라쏘:', np.sqrt(mean_squared_error(y_test, predict_lasso)))
print('릿지:', np.sqrt(mean_squared_error(y_test, predict_ridge)))
print('엘라:', np.sqrt(mean_squared_error(y_test, predict_elastic)))

일반: 0.7351277481981672
라쏘: 0.7400367400124828
릿지: 0.7351340103981142
엘라: 0.7372324657042565


R2 score

In [68]:
print('일반:', r2_score(y_test, predict_linear))
print('라쏘:', r2_score(y_test, predict_lasso))
print('릿지:', r2_score(y_test, predict_ridge))
print('엘라:', r2_score(y_test, predict_elastic))

일반: 0.5911695436410489
라쏘: 0.5856911861770583
릿지: 0.5911625783510761
엘라: 0.588825177197108
